# 실습 10: 드문 쪽을 놓치지 않게 만들기
- 상황: 어제 모델은 실제 불량 스물한 건 중 두 건만 잡았다
- 목표: 놓친 쪽을 줄이는 방법을 적용하고, 무엇을 내줬는지 함께 적는다

## Step 0. 어제 상태까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
for c in sensor_cols:
    df[c] = df[c].fillna(df[c].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

예측 = model.predict(X_test_scaled)

정확도 = accuracy_score(y_test, 예측) * 100
불량_예측_건수 = int((예측 == 1).sum())
그중_실제_불량_건수 = int(((예측 == 1) & (y_test == 1)).sum())

print("학습용:", X_train.shape, " 불량 건수:", y_train.sum())
print("시험용:", X_test.shape, " 불량 건수:", y_test.sum())
print("정확도:", round(정확도, 2), "%")
print("불량이라고 예측한 건수:", 불량_예측_건수)
print("그중 실제로 불량이었던 건수:", 그중_실제_불량_건수)

학습용: (1253, 50)  불량 건수: 83
시험용: (314, 50)  불량 건수: 21
정확도: 93.95 %
불량이라고 예측한 건수: 2
그중 실제로 불량이었던 건수: 2


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 드문 쪽을 다루는 말

| 말 | 뜻 |
|---|---|
| 클래스 불균형 | 한쪽이 지나치게 드문 상태. 우리 데이터는 불량이 약 6.6%뿐이다 |
| 클래스 가중치 | 드문 쪽 한 건을 여러 건만큼 무겁게 세도록 모델에 알려주는 설정 |
| 언더샘플링 | 많은 쪽을 줄여서 양쪽 수를 맞추는 방법. 데이터를 버리게 된다 |
| 오버샘플링 | 드문 쪽을 늘려서 양쪽 수를 맞추는 방법. 없던 기록을 만들어 넣게 된다 |
| 재현율 | 실제 불량 중 몇 %를 잡았나. 오늘 올리려는 숫자 |
| 정밀도 | 불량이라 한 것 중 몇 %가 진짜였나. 오늘 내주게 될 숫자 |

## Step 2. 무게를 다르게 주기

In [2]:
# 표준화와 모델을 한 줄로 묶어주는 도구들을 불러온다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# class_weight="balanced" - 드문 쪽 한 건을 그만큼 무겁게 세라는 뜻. 오늘 추가한 것은 이 한 조각뿐이다
가중치모델 = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)

# 학습용으로만 학습시킨다. 시험용은 여전히 건드리지 않는다
가중치모델.fit(X_train, y_train)

# 시험용 입력만 넣어 답을 받는다
가중치예측 = 가중치모델.predict(X_test)

print("불량이라고 예측한 건수:", 가중치예측.sum())
print("그중 진짜 불량:", ((가중치예측 == 1) & (y_test == 1)).sum())

불량이라고 예측한 건수: 78
그중 진짜 불량: 10


### 문법 노트 - 오늘 추가한 한 조각

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| class_weight="balanced" | 적은 쪽 한 건을 더 무겁게 세게 한다 | 그냥 두면 모델이 많은 쪽만 맞히고 만족해버린다 |
| max_iter=1000 | 답을 찾을 때까지 계산을 더 오래 하게 둔다 | 기본값으로는 다 못 찾고 멈췄다는 경고가 뜬다 |

## Step 3. 전후 숫자 비교하기

In [3]:
# 네 칸 표와 지표들을 계산해주는 도구를 불러온다
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 어제 예측과 오늘 예측을 나란히 놓고 같은 자로 잰다
for 이름, 예측 in [("손 안 댐", 예측), ("가중치", 가중치예측)]:
    # ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()
    print(f"[{이름}]")
    print("  정확도:", round((예측 == y_test).mean() * 100, 2), "%")
    print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
    print("  재현율:", round(recall_score(y_test, 예측), 3),
          "정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3),
          "F1:", round(f1_score(y_test, 예측), 3))

[손 안 댐]
  정확도: 93.95 %
  잡은 불량: 2 / 놓친 불량: 19 / 헛경보: 0
  재현율: 0.095 정밀도: 1.0 F1: 0.174
[가중치]
  정확도: 74.84 %
  잡은 불량: 10 / 놓친 불량: 11 / 헛경보: 68
  재현율: 0.476 정밀도: 0.128 F1: 0.202


## Step 4. 많은 쪽을 줄여서 해보기

In [4]:
# 학습용에서 양품만 불량 건수만큼 무작위로 남긴다 (많은 쪽인 양품만 줄인다)
양품_학습 = X_train[y_train == 0]
불량_학습 = X_train[y_train == 1]

양품_다운샘플 = 양품_학습.sample(n=len(불량_학습), random_state=42)

X_train_균형 = pd.concat([양품_다운샘플, 불량_학습])
y_train_균형 = y_train.loc[X_train_균형.index]

# 표준화는 줄인 학습용 기준으로 다시 맞춘다. 시험용은 여전히 손대지 않는다
균형_scaler = StandardScaler()
X_train_균형_scaled = 균형_scaler.fit_transform(X_train_균형)
X_test_균형_scaled = 균형_scaler.transform(X_test)

균형모델 = LogisticRegression(max_iter=1000)
균형모델.fit(X_train_균형_scaled, y_train_균형)

균형예측 = 균형모델.predict(X_test_균형_scaled)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 균형예측).ravel()

print("줄인 학습용 건수:", len(X_train_균형), " (불량", int(y_train_균형.sum()), "/ 양품", int((y_train_균형 == 0).sum()), ")")
print("정확도:", round((균형예측 == y_test).mean() * 100, 2), "%")
print("잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("재현율:", round(recall_score(y_test, 균형예측), 3),
      "정밀도:", round(precision_score(y_test, 균형예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 균형예측), 3))

줄인 학습용 건수: 166  (불량 83 / 양품 83 )
정확도: 73.25 %
잡은 불량: 14 / 놓친 불량: 7 / 헛경보: 77
재현율: 0.667 정밀도: 0.154 F1: 0.25


## Step 5. 전후 비교표

| 처리 | 정확도 | 잡은 불량 | 놓친 불량 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|---|
| 손 안 댐 | [93.95]% | [2] | [19] | [0] | [0.095] | [1.000] | [0.174] |
| 가중치 주기 | [74.84]% | [10] | [11] | [68] | [0.476] | [0.128] | [0.202] |
| 많은 쪽 줄이기 | [73.25]% | [14] | [7] | [77] | [0.667] | [0.154] | [0.250] |

## Step 6. 정직한 처리의 선

- 세 방법 모두 **학습용에만** 적용했다
- 시험용 314건은 처음 나눈 그대로 두었다 (불량 21건 그대로)
- 시험용을 손보면 점수는 올라가지만, 현장에 나가는 순간 그 점수는 없다

---
## 직접 해보기 (도전) - 시험지까지 손대면 어떻게 되나

- 상황: 학습용에만 손대라고 했는데, 시험용에도 손대면 점수가 어떻게 나올까
- 할 일: 시험용을 반반으로 맞춰놓고 같은 모델의 점수를 다시 잰다
- 결과물: 두 줄짜리 비교표 1개

In [5]:
# 시험용에서 양품만 불량 건수만큼 무작위로 남겨 반반짜리 시험용을 새로 만든다 (원래 X_test, y_test는 그대로 둔다)
양품_시험 = X_test[y_test == 0]
불량_시험 = X_test[y_test == 1]

양품_다운샘플_시험 = 양품_시험.sample(n=len(불량_시험), random_state=42)

X_test_반반 = pd.concat([양품_다운샘플_시험, 불량_시험])
y_test_반반 = y_test.loc[X_test_반반.index]

# Step 2에서 학습해둔 가중치모델을 그대로 쓴다 (재학습 없음)
가중치예측_반반 = 가중치모델.predict(X_test_반반)

비교표_시험용 = pd.DataFrame({
    "시험용": ["원래 시험용", "반반 시험용"],
    "건수": [len(y_test), len(y_test_반반)],
    "정확도(%)": [
        round((가중치예측 == y_test).mean() * 100, 2),
        round((가중치예측_반반 == y_test_반반).mean() * 100, 2),
    ],
    "재현율": [
        round(recall_score(y_test, 가중치예측), 3),
        round(recall_score(y_test_반반, 가중치예측_반반), 3),
    ],
    "정밀도": [
        round(precision_score(y_test, 가중치예측, zero_division=0), 3),
        round(precision_score(y_test_반반, 가중치예측_반반, zero_division=0), 3),
    ],
    "F1": [
        round(f1_score(y_test, 가중치예측), 3),
        round(f1_score(y_test_반반, 가중치예측_반반), 3),
    ],
})
비교표_시험용

,시험용,건수,정확도(%),재현율,정밀도,F1
0,원래 시험용,314,74.84,0.476,0.128,0.202
1,반반 시험용,42,69.05,0.476,0.833,0.606


### 시험지를 손대면

| 채점 방식 | 건수 | 정확도 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|
| 원래 시험용 | [314] | [74.84]% | [0.476] | [0.128] | [0.202] |
| 반반으로 맞춘 시험용 | [42] | [69.05]% | [0.476] | [0.833] | [0.606] |